# MLJAR AutoML 

MLJAR is an Automated Machine Learning framework. It is available as Python package with code at GitHub: https://github.com/mljar/mljar-supervised

The MLJAR AutoML can work in several modes:
- Explain - ideal for initial data exploration
- Perform - perfect for production-level ML systems
- Compete - mode for ML competitions under restricted time budget. By the default, it performs advanced feature engineering like golden features search, kmeans features, feature selection. It does model stacking.
- Optuna - uses Optuna to highly tune algorithms: Random Forest, Extra Trees, Xgboost, LightGBM, CatBoost, Neural Network. Each algorithm is tuned with `Optuna` hperparameters framework with selected time budget (controlled with `optuna_time_budget`). By the default feature engineering is not enabled (you need to manually swtich it on, in AutoML() parameter).


## Explain

The example useage of `Explain` with `MLJAR`:

```python

automl = AutoML(mode="Explain")
automl.fit(X, y)
```

The best choice to get initial information about your data. This mode will produce a lot of explanations for your data. All details can be viewed in the Notebook by calling the `automl.report()` method.


## Compete

The example useage of `Compete` with `MLJAR`:

```python

automl = AutoML(mode="Compete",
                total_time_limit=8*3600)
automl.fit(X, y)
```

That's it. It will train: Random Forest, Extra Trees, Xgboost, LightGBM, CatBoost, Neural Network, Ensemble, and stack all the models. Feature engineering will be applied (if enough training time). 


## Optuna

The example useage of `Optuna` with `MLJAR`:

```python

automl = AutoML(mode="Optuna", 
                optuna_time_budget=1800, 
                optuna_init_params={}, 
                algorithms=["LightGBM", "Xgboost", "Extra Trees"], 
                total_time_limit=24*3600)
automl.fit(X, y)
```

Description of parameters:
- `optuna_time_budget` - time budget for `Optuna` to tune each algorithm,
- `optuna_init_params` - if you have precomputed parameters for `Optuna` they can be passed here, then for already optimized models `Optuna` will not be used.
- `algorithms` - the algorithms that we will check,
- `total_time_limit` - the total time limit for AutoML training.

(In the `Optuna` mode, only first fold is used for model tuning.)

---

MLJAR GitHub: https://github.com/mljar/mljar-supervised

<img src="https://raw.githubusercontent.com/mljar/visual-identity/main/media/kaggle_banner_white.png" style="width: 70%;"/>

In [1]:
!pip install -q -U mljar-supervised

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.3.1 requires numpy>=1.20.0, but you have numpy 1.19.5 which is incompatible.
pdpbox 0.2.1 requires matplotlib==3.1.1, but you have matplotlib 3.4.0 which is incompatible.
mxnet 1.8.0.post0 requires graphviz<0.9.0,>=0.8.1, but you have graphviz 0.16 which is incompatible.
matrixprofile 1.1.10 requires protobuf==3.11.2, but you have protobuf 3.15.6 which is incompatible.
distributed 2021.3.1 requires cloudpickle>=1.5.0, but you have cloudpickle 1.3.0 which is incompatible.
autogluon-core 0.1.0 requires graphviz<0.9.0,>=0.8.1, but you have graphviz 0.16 which is incompatible.
autogluon-core 0.1.0 requires scipy==1.5.4, but you have scipy 1.6.1 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
from supervised.automl import AutoML # mljar-supervised

In [3]:
train = pd.read_csv("../input/digit-recognizer/train.csv")
test = pd.read_csv("../input/digit-recognizer/test.csv")

In [4]:
train.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
x_cols = train.columns[1:]
y_col = "label"

In [6]:
automl = AutoML(
    mode="Compete", 
    eval_metric="accuracy",
    total_time_limit=6*3600,
    validation_strategy={
        "validation_type": "kfold",
        "k_folds": 8
    },
    random_state=2021
)
automl.fit(train[x_cols], train[y_col])

Linear algorithm was disabled.
AutoML directory: AutoML_1
The task is multiclass_classification with evaluation metric accuracy
AutoML will use algorithms: ['Decision Tree', 'Random Forest', 'Extra Trees', 'LightGBM', 'Neural Network', 'Nearest Neighbors']
AutoML will stack models
AutoML will ensemble availabe models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'not_so_random', 'golden_features', 'insert_random_feature', 'features_selection', 'hill_climbing_1', 'hill_climbing_2', 'boost_on_errors', 'ensemble', 'stack', 'ensemble_stacked']
* Step simple_algorithms will try to check up to 3 models
1_DecisionTree accuracy 0.620952 trained in 60.74 seconds
2_DecisionTree accuracy 0.620952 trained in 39.45 seconds
3_DecisionTree accuracy 0.492524 trained in 38.55 seconds
* Step default_algorithms will try to check up to 4 models
4_Default_LightGBM accuracy 0.978095 trained in 2843.8 seconds
5_Default_NeuralNetwork accuracy 0.877524 trained in 619.99 seconds
6_Default_RandomFore

AutoML(eval_metric='accuracy', mode='Compete', random_state=2021,
       total_time_limit=21600,
       validation_strategy={'k_folds': 8, 'validation_type': 'kfold'})

In [7]:
preds = automl.predict(test[x_cols])
submission = pd.DataFrame({'ImageId':test.index+1, 'Label': preds})
submission.to_csv('1_submission.csv', index=False)

In [8]:
automl.report()

# Thank you!

<img src="https://raw.githubusercontent.com/mljar/visual-identity/main/media/robot_academy.png" style="width: 40%;"/>
